In [ ]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
import mlflow
import optuna

from sklearn.model_selection import (
    train_test_split,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, f1_score

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Bank-Customer-Churn-Prediction-Experiment")
mlflow.xgboost.autolog()
mlflow.lightgbm.autolog()

2025/09/11 08:17:16 INFO mlflow.tracking.fluent: Experiment with name 'Bank-Customer-Churn-Prediction-Experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1757553436595, experiment_id='1', last_update_time=1757553436595, lifecycle_stage='active', name='Bank-Customer-Churn-Prediction-Experiment', tags={}>

# Data preprocessing

In [3]:
data_path = "../data/Customer-Churn-Records.csv"


def clean_data(data_path):
    df = pd.read_csv(data_path)
    df.drop(columns=["RowNumber", "CustomerId", "Surname", "Complain"], inplace=True)
    df.head()

    return df


def split_data(df):
    # Train/val/test stratified split of ratio 0.8/0.1/0.1
    labels = df.Exited.values
    del df["Exited"]

    X_train, X_vtest, y_train, y_vtest = train_test_split(
        df, labels, test_size=0.2, random_state=seed, stratify=labels
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_vtest, y_vtest, test_size=0.5, random_state=seed, stratify=y_vtest
    )

    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
cat = [
    "Geography",
    "Gender",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "Satisfaction Score",
    "Card Type",
]

num = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary", "Point Earned"]


def preprocess_data(X_train, X_val, X_test):
    preprocessor = ColumnTransformer(
        [
            ("oh", OneHotEncoder(handle_unknown="ignore"), cat),
            ("scaler", StandardScaler(), num),
        ]
    )

    X_train = preprocessor.fit_transform(X_train)
    X_val = preprocessor.transform(X_val)
    X_test = preprocessor.transform(X_test)

    return X_train, X_val, X_test, preprocessor

In [ ]:
records = clean_data(data_path)
X_train, y_train, X_val, y_val, X_test, y_test = split_data(records)
X_train, X_val, X_test, pp = preprocess_data(X_train, X_val, X_test)

# Model Evaluation and Hyperparameters Tuning

In [32]:
sampler = optuna.samplers.TPESampler(seed=seed)

In [ ]:
def lgb_objective(trial):
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model", "lightgbm")
        dtrain = lgb.Dataset(X_train, label=y_train)
        dval = lgb.Dataset(X_val, label=y_val)

        params = {
            "objective": "binary",
            "metric": "binary_logloss",
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 2, 256),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "seed": seed,
        }

        gbm = lgb.train(params, dtrain, valid_sets=[dval])

        preds = gbm.predict(X_val)
        pred_labels = np.clip(np.rint(preds), 0, 1)

        f1 = f1_score(y_val, pred_labels)
        roc_auc = roc_auc_score(y_val, pred_labels)

        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_params(params)
    return roc_auc


study_lgb = optuna.create_study(direction="maximize", sampler=sampler)
study_lgb.optimize(lgb_objective, n_trials=100)

print("Number of finished trials:", len(study_lgb.trials))
print("Best trial:", study_lgb.best_trial.params)

[I 2025-09-11 08:27:26,817] A new study created in memory with name: no-name-7f7d60a0-021b-4c80-8765-6e903305070b


[LightGBM] [Info] Number of positive: 1630, number of negative: 6370
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000807 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1141
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203750 -> initscore=-1.363019
[LightGBM] [Info] Start training from score -1.363019
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

[I 2025-09-11 08:27:29,827] Trial 0 finished with value: 0.710673465366046 and parameters: {'lambda_l1': 2.294581980885925e-06, 'lambda_l2': 4.927325908265908, 'num_leaves': 229, 'feature_fraction': 0.897794369758139, 'bagging_fraction': 0.4861452683736408, 'bagging_freq': 3, 'min_child_samples': 10}. Best is trial 0 with value: 0.710673465366046.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 1630, number of negative: 6370
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000483 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1141
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203750 -> initscore=-1.363019
[LightGBM] [Info] Start training from score -1.36

[I 2025-09-11 08:27:33,183] Trial 1 finished with value: 0.705894669425559 and parameters: {'lambda_l1': 0.0023427694350445206, 'lambda_l2': 0.01046348996310111, 'num_leaves': 161, 'feature_fraction': 0.6216977061325458, 'bagging_fraction': 0.7295773959508784, 'bagging_freq': 5, 'min_child_samples': 18}. Best is trial 0 with value: 0.710673465366046.


...

[LightGBM] [Info] Number of positive: 1630, number of negative: 6370
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000491 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1141
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203750 -> initscore=-1.363019
[LightGBM] [Info] Start training from score -1.363019


[I 2025-09-11 08:29:12,684] Trial 98 finished with value: 0.7003029855158144 and parameters: {'lambda_l1': 0.0032323901882555686, 'lambda_l2': 2.8984613057077295e-06, 'num_leaves': 148, 'feature_fraction': 0.8348032690067133, 'bagging_fraction': 0.704873406324937, 'bagging_freq': 2, 'min_child_samples': 23}. Best is trial 89 with value: 0.7309710316287319.


[LightGBM] [Info] Number of positive: 1630, number of negative: 6370
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000440 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1141
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203750 -> initscore=-1.363019
[LightGBM] [Info] Start training from score -1.363019


[I 2025-09-11 08:29:13,677] Trial 99 finished with value: 0.7162035668538772 and parameters: {'lambda_l1': 0.00035117435687159114, 'lambda_l2': 5.5005191680749534e-08, 'num_leaves': 46, 'feature_fraction': 0.8077984786826025, 'bagging_fraction': 0.7306792397486782, 'bagging_freq': 1, 'min_child_samples': 39}. Best is trial 89 with value: 0.7309710316287319.


Number of finished trials: 100
Best trial: {'lambda_l1': 0.005443739635022496, 'lambda_l2': 3.2887227658298236e-06, 'num_leaves': 48, 'feature_fraction': 0.808137288097213, 'bagging_fraction': 0.7027256351103458, 'bagging_freq': 2, 'min_child_samples': 26}


In [15]:
study_lgb.best_value

0.7309710316287319

In [35]:
def xgb_objective(trial):
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model", "xgboost")
        train = xgb.DMatrix(X_train, label=y_train)
        valid = xgb.DMatrix(X_val, label=y_val)

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 5000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-9, 100.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-9, 100.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.1, 1.0),
            "max_depth": trial.suggest_int("max_depth", 1, 12),
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 1e-9, 0.5, log=True),
            "scale_pos_weight": trial.suggest_float(
                "scale_pos_weight", 1e-6, 500.0, log=True
            ),
            "seed": seed,
        }

        model = xgb.train(
            params, train, evals=[(valid, "validation")], early_stopping_rounds=300
        )

        preds = model.predict(valid)
        pred_labels = np.clip(np.rint(preds), 0, 1)

        f1 = f1_score(y_val, pred_labels)
        roc_auc = roc_auc_score(y_val, pred_labels)

        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_params(params)

    return roc_auc


study_xgb = optuna.create_study(direction="maximize", sampler=sampler)
study_xgb.optimize(xgb_objective, n_trials=200)

print("Number of finished trials:", len(study_xgb.trials))
print("Best trial:", study_xgb.best_trial.params)

[I 2025-09-11 09:05:33,708] A new study created in memory with name: no-name-c0c28fb2-495a-44e9-a212-5f42b3ea5b63


[0]	validation-rmse:0.45166
[1]	validation-rmse:0.45166
[2]	validation-rmse:0.45166
[3]	validation-rmse:0.45166
[4]	validation-rmse:0.45166
[5]	validation-rmse:0.45166
[6]	validation-rmse:0.45166
[7]	validation-rmse:0.45166
[8]	validation-rmse:0.45166
[9]	validation-rmse:0.45166


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [09:05:33] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-11 09:05:33,789] Trial 0 finished with value: 0.5 and parameters: {'n_estimators': 254, 'learning_rate': 0.18742210985555696, 'reg_lambda': 2.8702240018083644e-06, 'reg_alpha': 0.0003928959958815027, 'subsample': 0.9168098265334838, 'max_depth': 3, 'max_delta_step': 4, 'min_child_weight': 8, 'gamma': 9.77931372353297e-08, 'scale_pos_weight': 4.673539595873774e-06}. Best is trial 0 with value: 0.5.


🏃 View run nebulous-snail-973 at: http://localhost:5000/#/experiments/1/runs/a05421e532d64bceb48418fe21542489
🧪 View experiment at: http://localhost:5000/#/experiments/1
[0]	validation-rmse:0.44585
[1]	validation-rmse:0.44551
[2]	validation-rmse:0.44518
[3]	validation-rmse:0.44487
[4]	validation-rmse:0.44458
[5]	validation-rmse:0.44428
[6]	validation-rmse:0.44403
[7]	validation-rmse:0.44370
[8]	validation-rmse:0.44339
[9]	validation-rmse:0.44309


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [09:05:33] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-11 09:05:33,866] Trial 1 finished with value: 0.5 and parameters: {'n_estimators': 1520, 'learning_rate': 0.02101079931010356, 'reg_lambda': 16.852881837137915, 'reg_alpha': 0.7750401036522574, 'subsample': 0.6700633808593811, 'max_depth': 11, 'max_delta_step': 8, 'min_child_weight': 2, 'gamma': 0.05812245567496186, 'scale_pos_weight': 0.04917246552261189}. Best is trial 0 with value: 0.5.


...

🏃 View run puzzled-cub-313 at: http://localhost:5000/#/experiments/1/runs/ddd23ea3b74b4406a519898631435e32
🧪 View experiment at: http://localhost:5000/#/experiments/1
[0]	validation-rmse:0.68052
[1]	validation-rmse:0.63498
[2]	validation-rmse:0.61272
[3]	validation-rmse:0.60082
[4]	validation-rmse:0.58921
[5]	validation-rmse:0.58399
[6]	validation-rmse:0.57564
[7]	validation-rmse:0.57171
[8]	validation-rmse:0.56756
[9]	validation-rmse:0.56893


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [09:06:18] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-11 09:06:18,654] Trial 498 finished with value: 0.6631811015863631 and parameters: {'n_estimators': 1652, 'learning_rate': 0.3931007311253715, 'reg_lambda': 0.031238395951197238, 'reg_alpha': 1.082314591001148, 'subsample': 0.6097885782033612, 'max_depth': 4, 'max_delta_step': 6, 'min_child_weight': 10, 'gamma': 6.424696515401386e-07, 'scale_pos_weight': 19.947188821450734}. Best is trial 13 with value: 0.7923440733077151.


🏃 View run sneaky-fowl-300 at: http://localhost:5000/#/experiments/1/runs/36e77c2794344d9c91373d3e8fdc23f0
🧪 View experiment at: http://localhost:5000/#/experiments/1
[0]	validation-rmse:0.36735
[1]	validation-rmse:0.34953
[2]	validation-rmse:0.34172
[3]	validation-rmse:0.33819
[4]	validation-rmse:0.33419
[5]	validation-rmse:0.33365
[6]	validation-rmse:0.33278
[7]	validation-rmse:0.33038
[8]	validation-rmse:0.32805
[9]	validation-rmse:0.32756


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [09:06:18] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-11 09:06:18,746] Trial 499 finished with value: 0.7100453246625283 and parameters: {'n_estimators': 1570, 'learning_rate': 0.44132382428088246, 'reg_lambda': 0.015080003806921068, 'reg_alpha': 1.8673789926843514, 'subsample': 0.6250214362117369, 'max_depth': 3, 'max_delta_step': 8, 'min_child_weight': 10, 'gamma': 2.9014446607479415e-07, 'scale_pos_weight': 1.3999320019662855}. Best is trial 13 with value: 0.7923440733077151.


🏃 View run receptive-slug-39 at: http://localhost:5000/#/experiments/1/runs/d8a5e9dc0f29476989e030703b11eed4
🧪 View experiment at: http://localhost:5000/#/experiments/1
Number of finished trials: 500
Best trial: {'n_estimators': 1067, 'learning_rate': 0.40145384533052636, 'reg_lambda': 84.18994030652081, 'reg_alpha': 0.12460074360582617, 'subsample': 0.6711124746528365, 'max_depth': 5, 'max_delta_step': 10, 'min_child_weight': 10, 'gamma': 0.0002836927512130533, 'scale_pos_weight': 4.350498235429157}


In [ ]:
study_xgb.best_value

0.7923440733077151